# Занятие 7. Методы полей сил

In [ ]:
# 📥 Установка пакетов и библиотек
! pip install matplotlib
! pip install numpy
! pip install scipy
! pip install rdkit
! pip install networkx
! pip install plotly

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go
import os

from plotly.subplots import make_subplots

from scipy.optimize import curve_fit

from rdkit import Chem
from rdkit.Chem import Draw, AllChem

# 🦾 Глава 1. Силовое поле

Основная идея молекулярной механики - отказаться от явного описания электронов и заменить квантово-механическую задачу простыми параметрическими уравнениями: атомы представляются точечными массами с частичными зарядами. Структурно схожим атомам в разных молекулах присваивают одинаковые **типы атомов**. Тип атома зависит от атомного номера, гибридизации и химического окружения и обозначается цифровым или буквенным кодом.

**Силовое поле** - это набор уравнений, описывающих внутри- и межмолекулярные взаимодействия между атомами, вместе с набором параметров к ним. Уравнения содержат компоненты для разных видов взаимодействий: связей, валентных и торсионных углов, ван-дер-ваальсовых и электростатических взаимодействий. Значения параметров получают из квантово-механических расчётов или подгоняют под экспериментальные данные (плотности, энтальпии испарения, структурные данные).

## 1.1 Функциональная форма силового поля

Наиболее часто используемая форма силового поля (Amber, Gromacs, OpenMM):
$$V(r^N) = \sum_{i \in  bonds} \frac{1}{2} k_{bi} (l_i-l^0_i)^2 + \sum_{i \in  angles} \frac{1}{2} k_{\theta i}(\theta_i - \theta^0_i)^2 + \sum_{i\in torsions} \sum_n \frac{1}{2} V_i^n[1+\cos(n\omega_i-\gamma_i^n)]$$
$$+ \sum_{j=1}^{N-1}\sum_{i=j+1}^{N} 4\varepsilon_{ij}[(\frac{\sigma_{ij}}{r_{ij}})^{12}-(\frac{\sigma_{ij}}{r_{ij}})^{6}]+\frac{q_iq_j}{4\pi \varepsilon_0 r_{ij}}$$

Первые три терма - **связанные** (ковалентные) взаимодействия: двухатомные - связи, трёхатомные - валентные углы и четырёх атомные торсионные (диэдральные) углы, характеризующие вращение вокруг связи между вторым и третьим атомами. Последняя сумма - **несвязанные** взаимодействия для атомов, разделённых тремя и более связями или просто атомов разных молекул: ван-дер-ваальсовы (потенциал Леннарда-Джонса) и электростатические (закон Кулона). 

## 1.2 Типизация атомов

Например, в силовом поле GAFF2 (General Amber Force Field 2), предназначенном для органических молекул, атом углерода в зависимости от окружения получает один из следующих типов:

| Тип атома углерода (GAFF2) | Описание |
|---|---|
| c3 | sp³-углерод |
| c5, c6 | sp³-углерод в пяти- и шестичленных кольцах |
| c2 | sp²-углерод (алкены) |
| c | sp²-углерод карбонильной группы C=O |
| cs | sp²-углерод в C=S |
| c1 | sp-углерод (алкины, нитрилы) |
| ca | sp²-углерод в чисто ароматических кольцах (бензол) |
| cp / cq | ароматические углероды, соединяющие два кольца (бифенил) |
| cc / cd | sp²-углероды в гетероароматических и неароматических циклах (фуран, пиррол) |
| ce / cf | внутренние sp²-углероды сопряжённых цепей (бутадиен) |
| cg / ch | внутренние sp-углероды сопряжённых систем |
| cu / cv | sp²-углероды в трёх- и четырёхчленных кольцах |
| cx / cy | sp³-углероды в трёх- и четырёхчленных кольцах |
| cz | sp²-углерод гуанидиновой группы |

Парные типы (cc/cd, ce/cf, cg/ch) позволяют различать атомы углерода внутри сопряжённой системы.

Чем больше типов атомов содержит силовое поле, тем точнее поле описывает взаимодействия, но тем больше параметров нужно определить, и тогда возникают сложности с их согласованностью.

## 1.3 Топология: от графа молекулы к списку взаимодействий

Ранее для описания системы в методах квантовой химии нам было достаточно координат атомов: химические связи не задавались явно, а межатомные взаимодействия получались сами при решении электронной задачи. Сейчас же электронов в модели нет, поэтому информацию о том, какие атомы связаны и как они взаимодействуют, нужно задать явно.

С точки зрения молекулярной механики молекула может быть представлена в виде графа: вершины - атомы (с типами и частичными зарядами), рёбра - ковалентные связи. Из графа автоматически строятся списки взаимодействий.

Файл, в котором записаны эти списки вместе с типами атомов и параметрами, называется **топологией** (в GROMACS это файлы `.top`/`.itp`). 

In [ ]:
## Типы атомов MMFF94 на графах молекул

molecules = {
    'Уксусная кислота': 'CC(=O)O',
    'Толуол':           'Cc1ccccc1',
    'Цистеин':          'NC(CS)C(=O)O',
}

cmap = plt.get_cmap('tab20')
type_color = {}  # каждому типу — свой цвет из палитры, общий для всех молекул

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (name, smiles) in zip(axes, molecules.items()):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.Compute2DCoords(mol)

    # Типы атомов назначает RDKit — кодом силового поля MMFF94
    props = AllChem.MMFFGetMoleculeProperties(mol)
    types = [props.GetMMFFAtomType(i) for i in range(mol.GetNumAtoms())]
    for t in types:
        if t not in type_color:
            type_color[t] = cmap(len(type_color))

    G = nx.Graph()
    G.add_nodes_from(range(mol.GetNumAtoms()))
    G.add_edges_from((b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in mol.GetBonds())

    conf = mol.GetConformer()
    pos = {i: (conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y) for i in G.nodes}

    nx.draw(G, pos, ax=ax, node_color=[type_color[t] for t in types],
            labels={a.GetIdx(): a.GetSymbol() for a in mol.GetAtoms()}, font_color='black',
            font_size=9, node_size=700, width=2, edge_color='gray')
    ax.set_aspect('equal')
    ax.set_title(f"{name}: атомов - {G.number_of_nodes()}, типов - {len(set(types))}")

plt.tight_layout()
plt.show()

## 1.4 Ковалентные взаимодействия

Связанные слагаемые поля привязаны к топологии: на каждую строку списков связей, углов и диэдров приходится одно слагаемое, зависящее от одной внутренней координаты - длины связи $l$, валентного угла $\theta$ или диэдрального угла $\omega$. Эти термы локальны, их число растёт как $N$, и на фоне несвязанных взаимодействий, число которых растёт как $N^2$, их вклад в энергию рассчитывается быстро.

| Слагаемое | Атомов | Координата | Параметры |
|---|---|---|---|
| Связь | 2 | $l$ | $k_b$, $l_0$ | 
| Валентный угол | 3 | $\theta$ | $k_\theta$, $\theta_0$ |
| Торсионный угол | 4, последовательные | $\omega$ | $V_n$, $\gamma_n$ |
| Неправильный диэдр | 4, центр + соседи | $\xi$ | $k_\xi$, $\xi_0$ | 

### 1.4.1 Связи

Взаимодействие двух ковалентно связанных атомов может быть описано одним из трёх уравнений:

1. Гармонический потенциал - самое популярное уравнение, встречается почти во всех полях для биомолекул и жидкостей (AMBER, CHARMM, OPLS, GROMOS):
$$V(l) = \tfrac{1}{2} k_b (l-l_0)^2$$

2. Кубический потенциал (MM2, MM3), первая поправка на ангармонизм; $k_c < 0$ делает яму асимметричной:
$$V(l) = \tfrac{1}{2} k_b (l-l_0)^2\left[1 + k_c (l-l_0)\right]$$

3. Потенциал Морзе, единственный из трёх с конечной энергией диссоциации $D_e$:
$$V(l) = D_e\left(1 - e^{-a(l-l_0)}\right)^2$$

In [ ]:
## Поверхность потенциальной энергии для реакции диссоциации молекулы H₂
Distance_data = np.arange(0.30, 3.301, 0.05)
Energy_data   = np.array([
     191.899,   84.483,   14.100,  -32.641,  -63.673,  -83.953,  -96.712, -104.129, -107.713, -108.535,
    -107.363, -104.760, -101.141,  -96.813,  -92.008,  -86.903,  -81.633,  -76.305,  -71.001,  -65.784,
     -60.704,  -55.799,  -51.099,  -46.624,  -42.389,  -38.405,  -34.677,  -31.206,  -27.991,  -25.028,
     -22.311,  -19.829,  -17.574,  -15.534,  -13.696,  -12.048,  -10.574,   -9.262,   -8.098,   -7.068,
      -6.160,   -5.362,   -4.661,   -4.048,   -3.512,   -3.045,   -2.639,   -2.285,   -1.978,   -1.712,
      -1.481,   -1.282,   -1.109,   -0.960,   -0.831,   -0.719,   -0.623,   -0.540,   -0.468,   -0.406,
      -0.353])

def harmonic_potential(r, k, r0, b):
    return 0.5*k*(r - r0)**2 + b

def cubic_potential(r, k, kc, r0, b):
    return 0.5*k*(r - r0)**2 * (1 + kc*(r - r0)) + b

def morse_potential(r, D_e, a, r0):
    return D_e*(1 - np.exp(-a*(r - r0)))**2 - D_e

## Гармонический потенциал (0.65-0.85 Å)
near = (Distance_data > 0.64) & (Distance_data < 0.86)
harmonic_params, _ = curve_fit(harmonic_potential, Distance_data[near], Energy_data[near],
                               p0=[800, 0.74, -108])

## Кубический потенциал (0.55-1.10 Å)
mid = (Distance_data > 0.54) & (Distance_data < 1.11)
cubic_params, _ = curve_fit(cubic_potential, Distance_data[mid], Energy_data[mid],p0=[800, -1.0, 0.74, -108])

## Потенциал Морзе (по всей кривой)
morse_params, _ = curve_fit(morse_potential, Distance_data, Energy_data, p0=[110, 2.0, 0.74])

Distance_plot = np.linspace(0.3, 3.3, 400)

fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.set_title("Поверхность потенциальной энергии для реакции диссоциации молекулы H₂")
ax.axhline(0, color='0.6', lw=0.5, ls='--')
ax.plot(Distance_data, Energy_data, 'o', color='k', ms=2.5, label='CCSD/aug-cc-pVTZ')
ax.plot(Distance_plot, harmonic_potential(Distance_plot, *harmonic_params), label='гармонический')
ax.plot(Distance_plot, cubic_potential(Distance_plot, *cubic_params), label='кубический')
ax.plot(Distance_plot, morse_potential(Distance_plot, *morse_params), label='Морзе')
ax.set_xlim(0.3, 3.3); ax.set_ylim(-120, 60)
ax.set_xlabel('r(H–H), Å'); ax.set_ylabel('E, ккал/моль')
ax.legend(loc='upper right', ncol=2)
plt.show()

### 1.4.2 Валентные углы

Взаимодействие между тремя ковалентно связанными атомами может быть описано с использованием одного из следующих уравнений:

1. Гармонический потенциал: 
$$V(\theta) =  \frac{1}{2}  k_{\theta}(\theta - \theta_0)^2 $$

2. Потенциал Ури-Брэдли (где $s$ - расстояние между атомами 1 и 3): 

$$V(\theta, s) = \tfrac12 k_{\theta}(\theta - \theta_0)^2 + \tfrac12 k_{b} (s-s_0)^2 $$


In [ ]:
## Поверхность потенциальной энергии изгиба валентного угла в H₂O
Angle_data  = np.arange(30.0, 180.01, 2.5)
Energy_data = np.array([
     126.258,  114.358,  103.493,   93.671,   84.802,   76.767,   69.451,   62.751,   56.583,   50.878,
      45.588,   40.673,   36.103,   31.857,   27.918,   24.273,   20.911,   17.824,   15.005,   12.449,
      10.148,    8.099,    6.294,    4.729,    3.399,    2.296,    1.417,    0.754,    0.301,    0.052,
       0.000,    0.137,    0.455,    0.946,    1.601,    2.410,    3.363,    4.448,    5.655,    6.969,
       8.379,    9.869,   11.426,   13.034,   14.678,   16.341,   18.008,   19.660,   21.282,   22.856,
      24.364,   25.791,   27.120,   28.334,   29.420,   30.363,   31.153,   31.778,   32.231,   32.505,
      32.596,
])
r_OH = np.array([
    1.2632, 1.2019, 1.1573, 1.1237, 1.0978, 1.0773, 1.0609, 1.0475, 1.0364, 1.0271,
    1.0192, 1.0124, 1.0064, 1.0012, 0.9966, 0.9924, 0.9886, 0.9852, 0.9821, 0.9793,
    0.9767, 0.9742, 0.9720, 0.9698, 0.9679, 0.9660, 0.9644, 0.9628, 0.9613, 0.9598,
    0.9585, 0.9572, 0.9560, 0.9549, 0.9538, 0.9527, 0.9517, 0.9507, 0.9496, 0.9486,
    0.9477, 0.9467, 0.9458, 0.9449, 0.9440, 0.9431, 0.9422, 0.9413, 0.9404, 0.9396,
    0.9388, 0.9380, 0.9373, 0.9366, 0.9360, 0.9354, 0.9350, 0.9346, 0.9344, 0.9342,
    0.9341,
])

def harmonic_angle(theta, k, theta0, b):
    return 0.5*k*np.radians(theta - theta0)**2 + b

def urey_bradley(theta, k, k_ub, s0, b):
    s = 2*np.interp(theta, Angle_data, r_OH)*np.sin(np.radians(theta)/2)
    return 0.5*k*np.radians(theta - theta0)**2 + 0.5*k_ub*(s - s0)**2 + b

## Гармонический потенциал
near = (Angle_data >= 95) & (Angle_data <= 115)
harmonic_params, _ = curve_fit(harmonic_angle, Angle_data[near], Energy_data[near], p0=[100, 104, 0])
k_h, theta0, _ = harmonic_params

## Ури-Брэдли
mid = (Angle_data >= 70) & (Angle_data <= 150)
ub_params, _ = curve_fit(urey_bradley, Angle_data[mid], Energy_data[mid], p0=[50, 100, 1.5, 0],
                         bounds=([0, 0, 0.5, -5], [500, 2000, 3.0, 5]))

Angle_plot = np.linspace(Angle_data[0], Angle_data[-1], 400)

fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.set_title("Поверхность потенциальной энергии изгиба валентного угла в H₂O")
ax.plot(Angle_data, Energy_data, 'o', color='k', ms=2.5, label='CCSD/aug-cc-pVTZ')
ax.plot(Angle_plot, harmonic_angle(Angle_plot, *harmonic_params), label='гармонический')
ax.plot(Angle_plot, urey_bradley(Angle_plot, *ub_params), label='Ури–Брэдли')
ax.set_xlim(Angle_data[0], Angle_data[-1]); ax.set_ylim(-3, 130)
ax.set_xlabel('Угол H–O–H, °'); ax.set_ylabel('E, ккал/моль')
ax.legend(loc='upper right', ncol=1)
plt.show()

### 1.4.3 Торсионные углы

Торсионный (диэдральный) угол $\omega$ для цепочки атомов $i\!-\!j\!-\!k\!-\!l$ определяет угол между векторами нормали плоскостей $(i,j,k)$ и $(j,k,l)$, то есть поворот вокруг связи $j\!-\!k$. Поворот на $360°$ возвращает молекулу в исходное состояние, поэтому потенциал обязан быть периодическим:

$$V(\omega) = \sum_n \frac{V_n}{2}\left[1 + \cos(n\omega - \gamma_n)\right]$$

Кратность $n$ задаёт число минимумов при полном обороте, $V_n$ - высоту соответствующего барьера, фаза $\gamma_n$ (обычно $0°$ или $180°$) определяет положения минимумов. Типично $n = 3$ - вращение вокруг связи sp³–sp³ (три заслонённые конформации, как в этане), $n = 2$ - вращение вокруг sp²-sp² (плоские конформации), $n = 1$ обычно для *син-* и *анти-* конформаций (бутан, диметилпероксид). 

In [ ]:
## Торсионный угол
bond, angle = 1.53, 112.0

def chain(omega):
    a, w = np.radians(180 - angle), np.radians(omega)
    j = np.array([0.0, 0.0, 0.0])
    k = np.array([bond, 0.0, 0.0])
    i = j + bond*np.array([-np.cos(a), np.sin(a), 0.0])
    l = k + bond*np.array([np.cos(a), np.sin(a)*np.cos(w), np.sin(a)*np.sin(w)])
    return i, j, k, l

def arrow(start, vec, color, name, length=0.8):
    end = start + length*vec/np.linalg.norm(vec)
    return [go.Scatter3d(x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]], mode='lines',
                         line=dict(color=color, width=6), name=name),
            go.Cone(x=[end[0]], y=[end[1]], z=[end[2]], u=[vec[0]], v=[vec[1]], w=[vec[2]],
                    anchor='tip', sizemode='absolute', sizeref=0.25, showscale=False,
                    colorscale=[[0, color], [1, color]], name=name, showlegend=False)]

def traces(omega):
    i, j, k, l = chain(omega)
    P = np.array([i, j, k, l])
    arc_t = np.linspace(0, np.radians(omega), 40)            # дуга диэдра вокруг оси j–k
    arc = np.array([[bond/2, 0.5*np.cos(t), 0.5*np.sin(t)] for t in arc_t])
    n1 = np.cross(i - j, k - j)                               # нормаль к плоскости (i, j, k)
    n2 = np.cross(l - k, k - j)                               # нормаль к плоскости (j, k, l); угол между n1 и n2 равен ω
    O = np.array([-1.2, -1.2, -1.2])                          # начало триады осей
    out = [
        go.Scatter3d(x=P[:, 0], y=P[:, 1], z=P[:, 2], mode='lines+markers+text',
                     text=['i', 'j', 'k', 'l'], textposition='top center', textfont=dict(size=16),
                     marker=dict(size=9, color=['#00BFFF', '#000000', '#000000', '#FF674D']),
                     line=dict(color='#808080', width=8), name='атомы и связи'),
        go.Mesh3d(x=[i[0], j[0], k[0]], y=[i[1], j[1], k[1]], z=[i[2], j[2], k[2]],
                  color='#00BFFF', opacity=0.35, name='плоскость (i, j, k)', showlegend=True),
        go.Mesh3d(x=[j[0], k[0], l[0]], y=[j[1], k[1], l[1]], z=[j[2], k[2], l[2]],
                  color='#FF674D', opacity=0.35, name='плоскость (j, k, l)', showlegend=True),
        go.Scatter3d(x=arc[:, 0], y=arc[:, 1], z=arc[:, 2], mode='lines',
                     line=dict(color='#D64933', width=5), name=f'ω = {omega:.0f}°'),
    ]
    out += arrow((i + j + k)/3, n1, '#00BFFF', 'нормаль n₁')
    out += arrow((j + k + l)/3, n2, '#FF674D', 'нормаль n₂')
    return out

omegas = list(range(0, 361, 15))
fig = go.Figure(data=traces(60), frames=[go.Frame(data=traces(w), name=str(w)) for w in omegas])
fig.update_layout(
    width=950, height=500, margin=dict(l=0, r=0, t=30, b=0),
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
               aspectmode='data', camera=dict(eye=dict(x=1.6, y=1.2, z=0.9))),
    legend=dict(x=0.02, y=0.98),
    sliders=[dict(active=omegas.index(60), currentvalue=dict(prefix='ω = ', suffix='°'), pad=dict(t=10), steps=[dict(method='animate', label=str(w), args=[[str(w)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for w in omegas])],
)
fig.show()

In [ ]:
## Поверхность потенциальной энергии вращения торсионного угла H-C-C-H в молекуле этана
Dihedral_data = np.arange(0, 361, 10)
Energy_data   = np.array([
      2.906,   2.686,   2.109,   1.366,   0.665,   0.175,   0.000,   0.175,   0.666,   1.368,
      2.112,   2.688,   2.906,   2.687,   2.110,   1.365,   0.663,   0.174,   0.000,   0.174,
      0.664,   1.365,   2.110,   2.687,   2.906,   2.688,   2.111,   1.368,   0.666,   0.175,
      0.000,   0.175,   0.665,   1.365,   2.109,   2.686,   2.906,
])

def torsional_potential(phi_deg, V1, V2, V3):          # Три гармоники
    phi = np.radians(phi_deg)
    return (V1/2)*(1 + np.cos(phi)) + (V2/2)*(1 + np.cos(2*phi)) + (V3/2)*(1 + np.cos(3*phi))

def torsional_n3(phi_deg, V3):                          # Одна гармоника с n = 3
    return (V3/2)*(1 + np.cos(3*np.radians(phi_deg)))

fourier_params, _ = curve_fit(torsional_potential, Dihedral_data, Energy_data, p0=[0.0, 0.0, 3.0])
n3_params, _      = curve_fit(torsional_n3, Dihedral_data, Energy_data, p0=[3.0])

Dihedral_plot = np.linspace(0, 360, 721)

fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.set_title("Поверхность потенциальной энергии вращения торсионного угла H-C-C-H в молекуле этана")
ax.axhline(0, color='0.6', lw=0.5, ls='--')
ax.plot(Dihedral_data, Energy_data, 'o', color='k', ms=2.5, label='RI-MP2/cc-pVTZ')
ax.plot(Dihedral_plot, torsional_potential(Dihedral_plot, *fourier_params), label='ряд Фурье (n = 1, 2, 3)')
ax.plot(Dihedral_plot, torsional_n3(Dihedral_plot, *n3_params), '--', label='только n = 3')
ax.set_xlim(0, 360); ax.set_xticks(range(0, 361, 60)); ax.set_ylim(-0.3, 3.6)
ax.set_xlabel('Диэдральный угол H–C–C–H, °'); ax.set_ylabel('E, ккал/моль')
ax.legend(loc='upper right', ncol=3)
plt.show()

### 1.4.4 Неправильные торсионные углы

Связи и валентные углы не удерживают sp²-центр плоским. Три угла, близкие к $120°$, при карбонильном углероде совместимы и с плоской, и со слегка пирамидальной геометрией, а гармонические термы валентных углов описывают пирамидализацию недостаточно. Для этого вводят **неправильный (improper) диэдральный угол** - четвёрку атомов, не образующую цепочку последовательно соединённых атомов. 

1. Гармоническая форма по углу $\xi$ между плоскостями $(i,j,k)$ и $(j,k,l)$. В плоской молекуле $\xi = 0$:
$$V(\xi)=\tfrac{1}{2}k_\xi(\xi-\xi_0)^2$$

2. Периодическая форма, что и у обычного торсионного угла, с $n = 2$ и $\gamma = 180°$, в плоской молекуле $\omega = 180°$:
$$V(\omega)=\tfrac{V_2}{2}\left[1+\cos(2\omega-180°)\right]$$


In [ ]:
## Неправильный диэдр: центральный атом C и три соседа, плоскости (C, H1, H2) и (H1, H2, O)
r_CO, r_CH, a_HCO = 1.21, 1.10, 122.0

def molecule(alpha):
    a, h = np.radians(alpha), np.radians(a_HCO)
    C  = np.array([0.0, 0.0, 0.0])
    O  = r_CO*np.array([np.cos(a), 0.0, np.sin(a)])
    H1 = r_CH*np.array([np.cos(h),  np.sin(h), 0.0])
    H2 = r_CH*np.array([np.cos(h), -np.sin(h), 0.0])
    return C, O, H1, H2

def dihedral(p0, p1, p2, p3):
    b0, b1, b2 = p0 - p1, p2 - p1, p3 - p2
    b1 = b1/np.linalg.norm(b1)
    v = b0 - np.dot(b0, b1)*b1; w = b2 - np.dot(b2, b1)*b1
    return np.degrees(np.arctan2(np.dot(np.cross(b1, v), w), np.dot(v, w)))

alpha_grid = np.linspace(-100, 100, 2001)                   # связь ξ(alpha), чтобы задавать ξ напрямую
xi_grid = np.array([dihedral(*[molecule(a)[n] for n in (0, 2, 3, 1)]) for a in alpha_grid])
alpha_of_xi = lambda xi: np.interp(xi, xi_grid, alpha_grid)

def arrow(start, vec, color, name, length=0.7):
    end = start + length*vec/np.linalg.norm(vec)
    return [go.Scatter3d(x=[start[0], end[0]], y=[start[1], end[1]], z=[start[2], end[2]], mode='lines',
                         line=dict(color=color, width=6), name=name),
            go.Cone(x=[end[0]], y=[end[1]], z=[end[2]], u=[vec[0]], v=[vec[1]], w=[vec[2]],
                    anchor='tip', sizemode='absolute', sizeref=0.22, showscale=False,
                    colorscale=[[0, color], [1, color]], name=name, showlegend=False)]

def traces(xi):
    C, O, H1, H2 = molecule(alpha_of_xi(xi))
    i, j, k, l = C, H1, H2, O                                 # ξ = диэдр i–j–k–l = C–H1–H2–O, ось j–k = H1···H2
    axis_mid = (H1 + H2)/2
    arc_t = np.linspace(0, np.radians(xi), 40)
    arc = np.array([axis_mid + 0.45*np.array([np.cos(t), 0.0, np.sin(t)]) for t in arc_t])   # дуга вокруг оси H1–H2
    n1 = np.cross(i - j, k - j)                               # нормаль к плоскости (C, H1, H2)
    n2 = np.cross(l - k, k - j)                               # нормаль к плоскости (H1, H2, O); угол между n1 и n2 равен ξ
    bx, by, bz = [], [], []
    for X in (O, H1, H2):                                     # три связи от C, None разрывает линию
        bx += [C[0], X[0], None]; by += [C[1], X[1], None]; bz += [C[2], X[2], None]
    out = [
        go.Scatter3d(x=bx, y=by, z=bz, mode='lines', line=dict(color='#808080', width=8), name='связи'),
        go.Scatter3d(x=[H1[0], H2[0]], y=[H1[1], H2[1]], z=[H1[2], H2[2]], mode='lines',
                     line=dict(color='#808080', width=3, dash='dash'), name='ось H1···H2 (не связь)'),
        go.Scatter3d(x=[C[0], O[0], H1[0], H2[0]], y=[C[1], O[1], H1[1], H2[1]], z=[C[2], O[2], H1[2], H2[2]],
                     mode='markers+text', text=['C (i)', 'O (l)', 'H1 (j)', 'H2 (k)'], textposition='top center',
                     textfont=dict(size=14), marker=dict(size=[11, 10, 7, 7], color=['#000000', '#FF674D', '#00BFFF', '#00BFFF']),
                     name='атомы'),
        go.Mesh3d(x=[C[0], H1[0], H2[0]], y=[C[1], H1[1], H2[1]], z=[C[2], H1[2], H2[2]],
                  color='#00BFFF', opacity=0.35, name='плоскость (C, H1, H2)', showlegend=True),
        go.Mesh3d(x=[H1[0], H2[0], O[0]], y=[H1[1], H2[1], O[1]], z=[H1[2], H2[2], O[2]],
                  color='#FF674D', opacity=0.35, name='плоскость (H1, H2, O)', showlegend=True),
        go.Scatter3d(x=arc[:, 0], y=arc[:, 1], z=arc[:, 2], mode='lines',
                     line=dict(color='#D64933', width=5), name=f'ξ = {xi:.0f}°'),
    ]
    out += arrow((C + H1 + H2)/3, n1, '#00BFFF', 'нормаль n₁')
    out += arrow((H1 + H2 + O)/3, n2, '#FF674D', 'нормаль n₂')
    return out

xis = list(range(-50, 51, 5))
fig = go.Figure(data=traces(30), frames=[go.Frame(data=traces(x), name=str(x)) for x in xis])
fig.update_layout(
    width=950, height=500, margin=dict(l=0, r=0, t=30, b=0),
    scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
               aspectmode='data', camera=dict(eye=dict(x=1.4, y=1.5, z=0.8))),
    legend=dict(x=0.02, y=0.98),
    sliders=[dict(active=xis.index(30), currentvalue=dict(prefix='ξ = ', suffix='°'), pad=dict(t=10), steps=[dict(method='animate', label=str(x), args=[[str(x)], dict(mode='immediate', frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for x in xis])],
)
fig.show()

In [ ]:
## Сканирование валентных углов в молекуле формальдегида
HCO_scan = dict(
    HCO1 = np.array([
      97.50,  100.00,  102.50,  105.00,  107.50,  110.00,  112.50,  115.00,  117.50,  120.00,  122.50,
     125.00,  127.50,  130.00,  132.50,  135.00,  137.50,  140.00,  142.50,  145.00,  147.50,
    ]),
    HCO2 = np.array([
     128.56,  128.08,  127.54,  126.94,  126.29,  125.61,  124.88,  124.14,  123.35,  122.55,  121.69,
     120.87,  120.02,  119.15,  118.27,  117.37,  116.47,  115.56,  114.65,  113.73,  112.81,
    ]),
    HCH  = np.array([
     133.94,  131.92,  129.96,  128.06,  126.21,  124.39,  122.62,  120.86,  119.15,  117.45,  115.81,
     114.13,  112.48,  110.85,  109.23,  107.63,  106.03,  104.44,  102.85,  101.27,   99.69,
    ]),
    E    = np.array([
     15.296,  12.277,   9.594,   7.246,   5.235,   3.556,   2.207,   1.183,   0.479,   0.089,   0.008,
      0.229,   0.746,   1.553,   2.643,   4.011,   5.649,   7.550,   9.708,  12.115,  14.764,
    ]),
)
HCH_scan = dict(
    HCO1 = np.array([
     134.50,  133.25,  132.00,  130.75,  129.50,  128.25,  127.00,  125.75,  124.50,  123.25,  122.00,
     120.75,  119.50,  118.25,  117.00,  115.75,  114.50,  113.25,  112.00,  110.75,  109.50,
    ]),
    HCO2 = np.array([
     134.50,  133.25,  132.00,  130.75,  129.50,  128.25,  127.00,  125.75,  124.50,  123.25,  122.00,
     120.75,  119.50,  118.25,  117.00,  115.75,  114.50,  113.25,  112.00,  110.75,  109.50,
    ]),
    HCH  = np.array([
      91.00,   93.50,   96.00,   98.50,  101.00,  103.50,  106.00,  108.50,  111.00,  113.50,  116.00,
     118.50,  121.00,  123.50,  126.00,  128.50,  131.00,  133.50,  136.00,  138.50,  141.00,
    ]),
    E    = np.array([
     12.595,  10.118,   7.934,   6.033,   4.406,   3.046,   1.945,   1.097,   0.494,   0.130,   0.000,
      0.098,   0.417,   0.954,   1.702,   2.656,   3.811,   5.162,   6.703,   8.429,  10.333,
    ]),
)

def angle_energy(angles, k_HCO, theta0_HCO, k_HCH, theta0_HCH, b):
    hco1, hco2, hch = angles.T
    return (0.5*k_HCO*np.radians(hco1 - theta0_HCO)**2 + 0.5*k_HCO*np.radians(hco2 - theta0_HCO)**2
            + 0.5*k_HCH*np.radians(hch - theta0_HCH)**2 + b)

angles_all = np.vstack([np.column_stack([s['HCO1'], s['HCO2'], s['HCH']]) for s in (HCO_scan, HCH_scan)])
E_all      = np.concatenate([HCO_scan['E'], HCH_scan['E']])
angle_params, _ = curve_fit(angle_energy, angles_all, E_all, p0=[100, 122, 60, 116, 0])
k_HCO, theta0_HCO, k_HCH, theta0_HCH, _ = angle_params
rms = np.sqrt(np.mean((angle_energy(angles_all, *angle_params) - E_all)**2))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 2.6))
for ax, scan, key, xlabel in ((ax1, HCO_scan, 'HCO1', 'Угол H–C–O, °'), (ax2, HCH_scan, 'HCH', 'Угол H–C–H, °')):
    angles = np.column_stack([scan['HCO1'], scan['HCO2'], scan['HCH']])
    ax.plot(scan[key], scan['E'], 'o', color='k', ms=2.5, mfc='none', label='RI-MP2/cc-pVTZ')
    ax.plot(scan[key], angle_energy(angles, *angle_params), label='три гармонических угла')
    ax.set_xlabel(xlabel); ax.set_ylabel('E, ккал/моль')
plt.show()

In [ ]:
## Поверхность потенциальной энергии пирамидализации карбонильного атома углерода в H₂CO
Xi_data     = np.arange(-50, 51, 5)
Energy_data = np.array([
     35.936,  31.017,  25.961,  20.923,  16.075,  11.594,   7.653,   4.409,   1.993,   0.503,   0.000,
      0.503,   1.993,   4.409,   7.653,  11.594,  16.075,  20.923,  25.961,  31.017,  35.936,
])
Delta_data  = np.array([
      64.90,   60.63,   55.99,   50.93,   45.36,   39.22,   32.46,   25.07,   17.10,    8.68,    0.00,
      -8.68,  -17.10,  -25.07,  -32.46,  -39.22,  -45.36,  -50.93,  -55.99,  -60.63,  -64.90,
])
Angle_HCO1  = np.array([
      99.14,  101.63,  104.37,  107.29,  110.30,  113.26,  116.03,  118.44,  120.31,  121.51,  121.95,
     121.51,  120.31,  118.44,  116.03,  113.26,  110.30,  107.29,  104.37,  101.63,   99.14,
])
Angle_HCO2  = np.array([
      99.14,  101.63,  104.37,  107.29,  110.30,  113.26,  116.03,  118.44,  120.31,  121.51,  121.95,
     121.51,  120.31,  118.44,  116.03,  113.26,  110.30,  107.29,  104.37,  101.63,   99.14,
])
Angle_HCH   = np.array([
     138.46,  134.46,  130.77,  127.42,  124.47,  121.95,  119.87,  118.25,  117.09,  116.40,  116.11,
     116.40,  117.09,  118.25,  119.87,  121.95,  124.47,  127.42,  130.77,  134.46,  138.46,
])

## Вклад валентных углов с параметрами из предыдущей ячейки (k_HCO, theta0_HCO, k_HCH, theta0_HCH)
E_angles  = (0.5*k_HCO*np.radians(Angle_HCO1 - theta0_HCO)**2 + 0.5*k_HCO*np.radians(Angle_HCO2 - theta0_HCO)**2
             + 0.5*k_HCH*np.radians(Angle_HCH - theta0_HCH)**2)
E_angles -= E_angles[Xi_data == 0]
E_rest    = Energy_data - E_angles

def harmonic_improper(xi_deg, k):
    return 0.5*k*np.radians(xi_deg)**2

def periodic_improper(xi_deg, V2):
    omega = np.radians(180 + np.interp(xi_deg, Xi_data, Delta_data))
    return (V2/2)*(1 + np.cos(2*omega - np.pi))

harm_params, _ = curve_fit(harmonic_improper, Xi_data, E_rest, p0=[40])
per_params,  _ = curve_fit(periodic_improper, Xi_data, E_rest, p0=[20])

Xi_plot = np.linspace(-50, 50, 400)
angles_plot = lambda f: np.interp(Xi_plot, Xi_data, E_angles) + f

fig, ax = plt.subplots(figsize=(12.2, 4.6))
ax.set_title("Поверхность потенциальной энергии пирамидализации карбонильного атома углерода в H₂CO")
ax.plot(Xi_data, Energy_data, 'o', color='k', ms=2.5, mfc='none', label='RI-MP2/cc-pVTZ')
ax.plot(Xi_data, E_angles, ':', color='k', label='только валентные углы')
ax.plot(Xi_plot, angles_plot(harmonic_improper(Xi_plot, *harm_params)), label='валентные углы + гармонический improper')
ax.plot(Xi_plot, angles_plot(periodic_improper(Xi_plot, *per_params)), '--', label='валентные углы + периодический improper')
ax.set_xlim(-50, 50); ax.set_xticks(range(-50, 51, 10)); ax.set_ylim(-1, 45)
ax.set_xlabel('Неправильный диэдр ξ = C–H1–H2–O, °'); ax.set_ylabel('E, ккал/моль')
ax.legend(loc='upper center', ncol=2)
plt.show()

## 1.5 Несвязанные взаимодействия

Несвязанные термы описывают все пары атомов разных молекул и атомы одной молекулы на расстоянии трёх и более связей. Различают два типа взаимодействий: ван-дер-ваальсово (обменное отталкивание на коротких расстояниях и дисперсионное притяжение на средних) и электростатическое между частичными атомными зарядами.

### 1.5.1 Ван-дер-Ваальсовы взаимодействия

Ван-дер-ваальсово взаимодействие пары атомов может быть описано одним из потенциалов:

1. Потенциал Леннарда-Джонса
$$ V(r) = 4\varepsilon_{ij} \left[ \left( \frac{\sigma_{ij}}{r} \right)^{12} - \left( \frac{\sigma_{ij}}{r} \right)^6 \right] $$


2. Потенциал Букингема
$$ V(r) = A_{ij} \exp\left( -\frac{r}{\rho_{ij}} \right) - \frac{C_{ij}}{r^6}$$


3. Потенциал Морзе
$$ V(r) = D_{ij} \left(1 - e^{-a_{ij}(r-r_{ij}^0)}\right)^2$$

In [ ]:
## Несвязанные взаимодействия на примере ионной пары Li⁺F⁻
Distance_data = np.arange(1.5, 7.001, 0.1)
Energy_data   = np.array([
     -183.274,  -184.330,  -182.159,  -178.077,  -172.919,  -167.218,  -161.309,  -155.398,  -149.608,  -144.011,
     -138.645,  -133.528,  -128.668,  -124.063,  -119.709,  -115.597,  -111.716,  -108.055,  -104.601,  -101.342,
      -98.266,   -95.360,   -92.613,   -90.015,   -87.555,   -85.224,   -83.012,   -80.912,   -78.916,   -77.017,
      -75.209,   -73.484,   -71.839,   -70.268,   -68.766,   -67.328,   -65.951,   -64.631,   -63.364,   -62.147,
      -60.978,   -59.854,   -58.771,   -57.728,   -56.723,   -55.753,   -54.818,   -53.914,   -53.041,   -52.197,
      -51.381,   -50.591,   -49.829,   -49.092,   -48.383,   -47.706,
])

k_e = 332.0637                                              # e²/(4πε₀) в ккал·Å/моль
q_Li, q_F = +1.0, -1.0

def coulomb(r):                                              # заряды фиксированы, подгонять нечего
    return k_e*q_Li*q_F/r

def lennard_jones(r, sigma, eps):
    return 4*eps*((sigma/r)**12 - (sigma/r)**6)

def buckingham(r, A, rho, C):                                # exp-6: экспоненциальное отталкивание + дисперсия
    return A*np.exp(-r/rho) - C/r**6

def morse(r, D, a, r0):                                      # яма глубиной D при r₀, ноль на бесконечности
    return D*((1 - np.exp(-a*(r - r0)))**2 - 1)

E_vdw = Energy_data - coulomb(Distance_data)
well = (Distance_data >= 2.1) & (Distance_data <= 4)
lj_params,    _ = curve_fit(lennard_jones, Distance_data[well], E_vdw[well], p0=[1.9, 5.0], bounds=([1.0, 0.0], [4.0, 100.0]))
buck_params,  _ = curve_fit(buckingham,    Distance_data, E_vdw, p0=[5000, 0.3, 500], maxfev=20000)
morse_params, _ = curve_fit(morse,         Distance_data, E_vdw, p0=[5.0, 1.5, 2.5], maxfev=20000)

Distance_plot = np.linspace(1.5, 7.0, 400)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12.2, 6.4), sharex=True)
ax1.axhline(0, color='0.6', lw=0.5, ls='--')
ax1.plot(Distance_data, Energy_data, 'o', color='k', ms=2.5, mfc='none', label='CCSD/aug-cc-pVTZ')
ax1.plot(Distance_plot, coulomb(Distance_plot), ':', color='k', label='только кулон, q = ±1')
ax1.plot(Distance_plot, coulomb(Distance_plot) + lennard_jones(Distance_plot, *lj_params), label='кулон + Леннард-Джонс')
ax1.plot(Distance_plot, coulomb(Distance_plot) + buckingham(Distance_plot, *buck_params), '--', label='кулон + Букингем')
ax1.plot(Distance_plot, coulomb(Distance_plot) + morse(Distance_plot, *morse_params), label='кулон + Морзе')
ax1.set_ylim(-220, 100); ax1.set_ylabel('E, ккал/моль')
ax1.legend(loc='lower right', ncol=3)

ax2.axhline(0, color='0.6', lw=0.5, ls='--')
ax2.plot(Distance_data, E_vdw, 'o', color='k', ms=2.5, mfc='none', label='E − E(кулон)')
ax2.plot(Distance_plot, lennard_jones(Distance_plot, *lj_params), label='Леннард-Джонс')
ax2.plot(Distance_plot, buckingham(Distance_plot, *buck_params), '--', label='Букингем')
ax2.plot(Distance_plot, morse(Distance_plot, *morse_params), label='Морзе')
ax2.axvline(1.8, color='0.6', lw=0.5, ls=':')
ax2.set_xlim(1.5, 7.0); ax2.set_ylim(-10, 10)
ax2.set_xlabel('r(Li$^+$–F$^-$), Å'); ax2.set_ylabel('E(ВДВ), ккал/моль')
ax2.legend(loc='upper right', ncol=4)
plt.show()

### 1.5.2 Правила комбинирования

При наличии большого количества типов атомов становится трудно добавлять и изменять параметры нековалентных взаимодействий в файлах силового поля для каждой пары типов атомов. Поэтому каждому типу атомов присваивают свой коэффициент несвязанных взаимодействий  $\sigma_i$ и $\varepsilon_i$ и при построении матрицы параметров используют одно из следующих правил:

1. Арифметическое среднее (правило Лоренца-Бертло):
$$ \sigma_{ij} = \frac{1}{2} (\sigma_{i} + \sigma_{j}) \\
\varepsilon_{ij} = \left( \varepsilon_{i} \varepsilon_{j} \right)^{1/2} $$

2. Геометрическое среднее:
$$ \sigma_{ij} = \left( \sigma_{i} \sigma_{j} \right)^{1/2} \\
\varepsilon_{ij} = \left( \varepsilon_{i} \varepsilon_{j} \right)^{1/2} $$

### 1.5.3 Электростатические взаимодействия

Электростатическое взаимодействие между парой несвязанных атомамов с зарядами $q_i$ и $q_j$ может быть описано по закону Кулона:

$$V(r) = \frac{q_i q_j}{4\pi\varepsilon_0\, r}$$

В ионной паре Li⁺···F⁻ использовались целые заряды. В молекуле же атомы имеют дробный, **частичный заряд** $q_i$. Частичные атомные заряды это параметры модели, которые подбирают так, чтобы набор зарядов воспроизводил электростатический потенциал молекулы вокруг неё (тот самый ESP из занятия 6), или, чтобы воспроизводить физико-химические свойства системы.

У частичных зарядов три свойства, которых нет у остальных параметров поля:

* они **не привязаны к типу атома**: карбонильный углерод в уксусной кислоте и в цистеине могут иметь один и тот же тип атома, но заряды у них могут различаться, поэтому заряды задают для каждой молекулы или остатка отдельно, а не в таблице типов.
* **сумма зарядов по молекуле (или в остатке) обязана быть целым числом**

In [ ]:
## Модели воды
water_models = {
    'QM, Малликен': dict(r=0.9591, theta=103.52, q_H=+0.2351, q_O=-0.4702, mu_qm=1.94),
    'SPC/E':        dict(r=1.0000, theta=109.47, q_H=+0.4238, q_O=-0.8476),
    'TIP3P':        dict(r=0.9572, theta=104.52, q_H=+0.4170, q_O=-0.8340),
    'TIP4P/2005':   dict(r=0.9572, theta=104.52, q_H=+0.5564, q_O=0.0, q_M=-1.1128, d=0.1546),
    'TIP5P':        dict(r=0.9572, theta=104.52, q_H=+0.2410, q_O=0.0, q_L=-0.2410, d=0.70, theta_L=109.47),
}
e_A_to_D = 4.8032                                           # 1 e·Å = 4.8032 Д

def sites(m):
    """Список узлов модели: (подпись, x, y, z, заряд). O в начале координат, биссектриса HOH вдоль +y."""
    h = np.radians(m['theta']/2)
    s = [('O', 0.0, 0.0, 0.0, m['q_O']),
         ('H', +m['r']*np.sin(h), m['r']*np.cos(h), 0.0, m['q_H']),
         ('H', -m['r']*np.sin(h), m['r']*np.cos(h), 0.0, m['q_H'])]
    if 'q_M' in m:                                         
        s.append(('M', 0.0, m['d'], 0.0, m['q_M']))
    if 'q_L' in m:  
        l = np.radians(m['theta_L']/2)
        s += [('L', 0.0, -m['d']*np.cos(l), +m['d']*np.sin(l), m['q_L']),
              ('L', 0.0, -m['d']*np.cos(l), -m['d']*np.sin(l), m['q_L'])]
    return s

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
size = {'O': 900, 'H': 380, 'M': 140, 'L': 140}
for ax, (name, m) in zip(axes, water_models.items()):
    S = sites(m)
    for lab, x, y, z, q in S[1:]:                           # связи и штрихи к виртуальным узлам
        ax.plot([0, x], [0, y], '-' if lab == 'H' else ':', color='0.6', lw=2 if lab == 'H' else 1.2, zorder=1)
    for lab, x, y, z, q in S:
        ax.scatter(x, y, s=size[lab], c=[q], cmap='RdBu_r', vmin=-1.2, vmax=1.2, edgecolors='0.3', linewidths=0.8, zorder=2)
    ax.text(0, 0.30, f"O {S[0][4]:+.3f}", ha='center', va='center', fontsize=11)
    for lab, x, y, z, q in S[1:3]:
        ax.text(x, y + 0.28, f"H {q:+.3f}", ha='center', va='center', fontsize=11)
    if 'q_M' in m:
        ax.text(0.25, m['d'], f"M {m['q_M']:+.3f}", ha='left', va='center', fontsize=11)
    if 'q_L' in m:
        ax.text(0, S[3][2] - 0.28, f"2 × L {m['q_L']:+.3f}\n(вне плоскости)", ha='center', va='center', fontsize=10)
    mu = e_A_to_D*np.linalg.norm(sum(q*np.array([x, y, z]) for _, x, y, z, q in S))
    title = f"{name}\nμ по зарядам = {mu:.2f} Д" + (f"\nμ(QM) = {m['mu_qm']:.2f} Д" if 'mu_qm' in m else "")
    ax.set_title(title, fontsize=12)
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.3, 1.4)
    ax.axis('off')
plt.show()

Простые модели воды не воспроизводят все свойства сразу. Ниже температура плавления льда $T_{пл}$ и температура максимума плотности $T_{макс}$ (https://doi.org/10.1039/c1cp22168j):

| Модель | $T_{пл}$, K | $T_{макс}$, K |
|---|---|---|
| TIP3P | 146 | 182 |
| SPC | 190 | 228 |
| SPC/E | 215 | 241 |
| TIP4P | 232 | 253 |
| TIP4P/2005 | 252 | 278 |
| TIP5P | 274 | 285 |
| TIP4P/Ice | 272 | 295 |
| Эксперимент | 273.15 | 277 |

## 1.6 Применение методов молекулярной механики

Собранное силовое поле $V(r^N)$ позволяет получить энергию системы и её градиент по координатам атомов для любой геометрии на порядки быстрее квантовой химии, что делает молекулярную механику идеальным инструментом для задач: поиск минимумов (оптимизация геометрии, конформационный поиск), интегрирование уравнений движения (молекулярная динамика, глава 2), оценка энергии связывания лиганда с белком или белка с белком (докинг, занятие 9) и расчёт свободных энергий. 

| Поле | Для чего |
|---|---|
| AMBER ff14SB / ff19SB | белки |
| AMBER OL24 (ДНК) / χOL3 (РНК) | нуклеиновые кислоты (ДНК и РНК)|
| GAFF2 | органические молекулы и лиганды, совместимо с AMBER |
| CHARMM36 / CHARMM36m | белки, нуклеиновые кислоты, липиды, углеводы |
| CGenFF | органические молекулы и лиганды, совместимо с CHARMM (аналог GAFF) |
| OPLS-AA | органические жидкости, белки |
| Rosetta REF2015 | предсказание структуры и дизайн белков |
| MMFF94 / UFF| малые молекулы; встроены в RDKit |
| GFN-FF | малые молекулы; используется в xtb и CREST |


### 1.6.1 Оптимизация геометрии

Оптимизация геометрии в молекулярной механике с ипользованием силового поля выполняется значительно быстрее, чем в квантово-химических расчётах, где вычисление энергии и её производных связано с решением электронной задачи. Дополнительно производные по координатам атомов в молекулярной механике можно вычислять аналитически для отдельных составляющих потенциальной энергии.

Например, для гармонического потенциала растяжения связи: 
$$
V(l)=\frac{1}{2}k_b(l-l_0)^2
$$
производная по длине связи равна:
$$
\frac{dV(l)}{dl}= k_b(l-l_0).
$$

Таким образом, вычисление градиента энергии выполняется быстро, что делает метод молекулярной механики оптимальным для использования в *gradient-based* методах оптимизации геометрии (steepest descent, CG, L-BFGS). 

### 1.6.2 Конформационный поиск

Оптимизация геометрии позволяет найти локальный минимум потенциальной энергии, однако для одной молекулы таких минимумов может быть несколько. Они соответствуют конформациям - различным пространственным расположениям атомов для молекул с одинаковой топологией. Чтобы обнаружить возможные конформации и оценить их заселённости, используют конформационный поиск (conformational search).

Конформационный поиск представляет собой процедуру генерации различных начальных геометрий молекулы с последующей оптимизацией каждой из них. Начальные структуры можно получать случайным изменением торсионных углов, вращением отдельных фрагментов, случайным возмущением координат или с помощью коротких молекулярно-динамических симуляций (как это реализовано в программе CREST). После оптимизации геометрии каждая структура оказывается в одном из локальных минимумов потенциальной энергии.

Поскольку разные начальные геометрии могут приводить к одному и тому же минимуму, полученные структуры необходимо сгруппировать. Для этого в качестве метрики можно использовать среднеквадратичное отклонение координат (root mean square deviation, RMSD). Для двух структур, содержащих $N$ атомов, оно определяется как:

$$
\mathrm{RMSD}=
\sqrt{\frac{1}{N}\sum_{i=1}^{N}
\left|\mathbf r_i^{(A)}-\mathbf r_i^{(B)}\right|^2},
$$
где $\mathbf r_i^{(A)}$ и $\mathbf r_i^{(B)}$ это координаты $i$-го атома в двух структурах $A$ и $B$.
Перед вычислением RMSD структуры обычно совмещают оптимальными операциями переноса и вращения, чтобы исключить различия, обусловленные положением молекулы в пространстве.

# 🚀 Глава 2. Молекулярная динамика

## 2.1 Уравнения движения

**Молекулярная динамика** - вычислительный метод, в котором траектории атомов получают численным интегрированием второго закона Ньютона:

$$m_i\,\ddot{\mathbf{r}}_i(t) = \mathbf{F}_i(t) = -\nabla_{\mathbf{r}_i} V\!\left(\mathbf{r}^N\right)$$

где $m_i$ - масса $i$-го атома, $\mathbf r_i(t)$ - его координата в момент времени $t$, $\mathbf F_i(t)$ - действующая на него сила, $V(\mathbf r^N)$ - потенциальная энергия системы, а $\mathbf r^N=(\mathbf r_1,\ldots,\mathbf r_N)$ обозначает совокупность координат всех $N$ атомов.

В молекулярной механике потенциальная энергия определяется выбранным силовым полем, поэтому силы можно вычислять непосредственно из его аналитических производных. Зная силы и текущие координаты и скорости атомов, можно определить их ускорения и рассчитать положение системы в следующий момент времени.

## 2.2 Интеграторы

Аналитически уравнения Ньютона для трёх и более взаимодействующих тел не решаются, поэтому их интегрируют численно с шагом $\Delta t$. 

### 2.2.1 Интегратор Верле
Запишем разложение в ряд Тейлора для положения частицы в моменты времени $t \pm \Delta t$:

$$\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\Delta t + \tfrac{1}{2}\mathbf{a}(t)\Delta t^2 + O(\Delta t^3)$$

$$\mathbf{r}(t-\Delta t) = \mathbf{r}(t) - \mathbf{v}(t)\Delta t + \tfrac{1}{2}\mathbf{a}(t)\Delta t^2 - O(\Delta t^3)$$

При сложении уравнений термы с нечётными степенями $\Delta t$ сокращаются:

$$\boxed{\mathbf{r}(t+\Delta t) = 2\mathbf{r}(t) - \mathbf{r}(t-\Delta t) + \mathbf{a}(t)\,\Delta t^2}$$

Недостаток: скорость не получается явно. Нам она нужна для расчёта кинетической энергии частиц и термодинамических параметров системы (температуры, давления). Восстановить её можно из из интегратора Верле $[\mathbf{r}(t+\Delta t)-\mathbf{r}(t-\Delta t)]/2\Delta t$, но лишь с опозданием на шаг и с потерей точности при вычитании почти равных координат.

### 2.2.2 Интегратор velocity-Verlet

Интегратор velocity-Verlet позволяет одновременно обновлять координаты и скорости атомов, используя их значения в текущий момент времени и ускорения в начале и конце временного шага. Один шаг интегрирования состоит из трёх стадий:

1. **Обновление скорости на половину шага:**

$$ \mathbf{v}\left(t+\frac{\Delta t}{2}\right) = \mathbf{v}(t)+\frac{1}{2}\mathbf{a}(t)\Delta t $$

2. **Обновление координат на полный шаг:**

$$ \mathbf{r}(t+\Delta t) = \mathbf{r}(t)+ \mathbf{v}\left(t+\frac{\Delta t}{2}\right)\Delta t $$

3. **Обновление скорости на вторую половину шага:**

$$ \mathbf{v}(t+\Delta t) = \mathbf{v}\left(t+\frac{\Delta t}{2}\right) +\frac{1}{2}\mathbf{a}(t+\Delta t)\Delta t $$

На втором этапе вычисляют новые координаты, после чего по ним определяют силы и ускорения в момент времени $t+\Delta t$. Эти ускорения используют на третьем этапе для завершения обновления скоростей. Используя алгоритм velocity-Verlet, можно на каждом шаге рассчитывать кинетическую энергию системы: 

$$E_{кин}=\sum_{i=1}^{N}\frac{m_i|\mathbf{v}_i|^2}{2} $$

### 2.2.3 Шаг интегрирования и ограничения связей

Шаг интегрирования $\Delta t$ определяет временной интервал между шагами расчёта сил и обновления координат атомов. Его выбор представляет собой компромисс между точностью численного интегрирования и вычислительными затратами, так как уменьшение шага повышает точность, но увеличивает число шагов, нужных для заданного промежутка времени.

При выборе шага интегрирования необходимо учитывать характерные временные масштабы движения атомов. Наиболее быстрые движения в молекулярных системах связаны с колебаниями химических связей, особенно связей с участием атомов водорода. Чтобы корректно описывать такие движения, шаг интегрирования должен быть значительно меньше периода наиболее быстрых колебаний.

Слишком большой шаг приводит к накоплению ошибок интегрирования и в конечном итоге к численной нестабильности системы. Атомы могут приобретать нефизически высокие скорости, а геометрия молекул - существенно искажаться.

В классической молекулярной динамике обычно используют следующие значения шага интегрирования:
* **от 0.5 до 1 фс** применяется при уравновешивании системы в плохих начальных координатах. 
* **2 фс** является самым распространённым выбором шага интегрирования для биомолекулярных систем.

Уменьшение шага интегрирования не устраняет ошибки самой модели взаимодействий, а лишь уменьшает ошибки численного интегрирования уравнений движения. Даже идеально проинтегрированная траектория не гарантирует физической достоверности, если используемое силовое поле недостаточно точно описывает систему.

## 2.3 Ансамбли, термостаты и баростаты 

При проведении молекулярной динамики необходимо определить, какие параметры системы остаются постоянными, а какие могут изменяться в ходе моделирования. Набор таких условий задаётся термодинамическим ансамблем.

### 2.3.1 Микроканонический ансамбль (NVE)

В моделировании системы в микроканоническом ансамбле постоянны число частиц $N$, объём системы $V$ и полная энергия (сумма кинетической и потенциальной энергий) $E_{полн}$. Система не обменивается энергией с окружающей средой, и полная энергия системы равняется константе в ходе всей симуляции:

$$ E_{полн}=\mathrm{const} $$

В реальном численном моделировании энергия может немного колебаться из-за погрешности интегрирования. Систематический дрейф энергии могут указывать на выбор слишком большого шага интегрирования или другие численные проблемы. Ансамбль NVE используют, в частности, для проверки качества интегрирования уравнений движения и изучения динамики изолированной системы.

### 2.3.2 Канонический ансамбль (NVT)

В каноническом ансамбле постоянны число частиц $N$, объём $V$ и температура $T$. В отличие от NVE, система может обмениваться энергией с окружающей средой, что в молекулярной динамике обычно моделируют с использованием термостата. Алгоритм термостата изменяет динамику системы таким образом, чтобы её статистические свойства соответствовали заданной температуре. 

Например, термостат Берендсена удерживает систему вблизи заданной температуре $T_0$, масштабируя скорости на коэффициент $\lambda$ на каждом шаге моделирования:

$$\mathbf{v_{ref}}(t) \;\leftarrow\; \lambda\,\mathbf{v}(t), \qquad \lambda = \sqrt{1 + \frac{\Delta t}{\tau_T}\!\left(\frac{T_0}{T(t)} - 1\right)}$$

где $\tau_T$ определяет характерное время релаксации температуры (чем оно меньше, тем сильнее термостат воздействует на систему и тем быстрее температура приближается к заданному значению), а мгновенная температура рассчитывается через кинетическую энергию:

$$T(t) = \frac{\displaystyle\sum_i m_i v_i^2}{N_fk_B}$$
где $N_f$ - число степеней свободы системы. Термостат Берендсена обеспечивает быстрое приближение температуры к целевому значению и часто используется на этапе предварительной стабилизации системы. Однако он не воспроизводит правильные флуктуации кинетической энергии канонического ансамбля, поэтому его не следует считать строгим методом моделирования NVT. Для равновесных расчётов используют термостаты Нозе-Гувера или v-rescale.

### 2.3.3 Изобарно-изотермический ансамбль (NPT)

В изобарно-изотермическом ансамбле постоянны число частиц $N$, давление $P_0$ и температура $T_0$. В отличие от ансамбля NVT, объём системы может изменяться, позволяя ей адаптировать плотность к заданному давлению. Для поддержания постоянных температуры и давления используют два алгоритма: термостат регулирует температуру посредством изменения динамики атомов, а **баростат** управляет объёмом расчётной ячейки. Изменение объёма приводит к изменению координат частиц и плотности системы.

Давление в молекулярной динамике связано как с движением частиц, так и с их взаимодействиями. Для системы в периодических граничных условиях  с тремя поступательными степенями свободы на частицу его можно выразить через вириальное уравнение:

$$ P=\frac{Nk_BT}{V} +\frac{1}{3V}\sum_i \mathbf{r}_i\cdot\mathbf{F}_i $$

где $V$ это объём системы, $\mathbf{r}_i$ - координата атома относительно выбранного начала отсчёта, а $\mathbf{F}_i$ - результирующая сила, действующая на него. Если мгновенное давление отличается от заданного, баростат изменяет объём системы так, чтобы среднее давление со временем приближалось к $P_0$.

## 2.4 Периодические граничные условия и дальнодействующие взаимодействия

При моделировании молекулярных систем число атомов ограничено вычислительными ресурсами. Однако реальные системы, например растворы или биополимеры в среде растворителя, содержат значительно больше частиц. Чтобы уменьшить влияние искусственных границ расчётной системы, используют периодические граничные условия (periodic boundary conditions, PBC).

Систему помещают в ячейку, которая виртуально повторяется во всех направлениях, образуя бесконечную периодическую структуру. Если атом покидает ячейку через одну из её границ, его периодический образ входит через противоположную границу. При этом физически моделируется одна ячейка, а не бесконечное количество независимых систем.

Периодические граничные условия позволяют моделировать объёмные системы без свободных поверхностей. Однако для корректного расчёта взаимодействий необходимо учитывать два обстоятельства: 
1. Размер ячейки должен быть достаточным, чтобы избежать нежелательного взаимодействия системы с собственными периодическими образами. 
2. Различные типы межатомных взаимодействий требуют разных способов расчёта: короткодействующие взаимодействия (~$\frac{1}{r^6}$) можно рассчитывать с помощью отсечки, тогда как для дальнодействующих кулоновских взаимодействий (~$\frac{1}{r}$) нужны специальные методы (§ 2.4.2-2.4.4).

### 2.4.1 Выбор бокса для моделирования

Для компактной молекулы в вакууме периодические образы должны находиться достаточно далеко друг от друга, чтобы взаимодействие между ними не влияло существенно на результаты. Например, белок в водном растворе необходимо окружить слоем воды и обеспечить достаточное расстояние между его периодическими образами. Если ячейка слишком мала, белок может взаимодействовать со своими образами, что приводит к искусственным ограничениям его конформационной динамики.

В молекулярной динамике выбор формы и размера расчётной ячейки влияет на количество моделируемых частиц и, как следствие, на вычислительную сложность. Наиболее распространены следующие варианты:

* **Кубическая ячейка**. Все рёбра имеют одинаковую длину, а углы между ними равны $90^\circ$. Такая ячейка проста в настройке и удобна для моделирования объёмных систем, например растворов.
* **Прямоугольная ячейка**. Все углы равны $90^\circ$, но длины рёбер могут различаться. Её используют, когда система имеет вытянутую или плоскую форму, например при моделировании мембранных белков, поверхностей или длинных биомолекулярных комплексов.
* **Усечённый октаэдр**. Компактнее окружает глобулярную молекулу растворителем по сравнению с кубической ячейкой при сопоставимом минимальном расстоянии до периодических образов.

In [ ]:
cv=np.array([[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]],float)
rv=np.array([[0,0,0],[2.5,0,0],[2.5,1.4,0],[0,1.4,0],[0,0,1],[2.5,0,1],[2.5,1.4,1],[0,1.4,1]],float)/1.5
ov=[]
for x,y,z in [(0,1,2),(0,1,-2),(0,-1,2),(0,-1,-2)]:
    ov+=[[x,y,z],[x,z,y],[y,x,z],[y,z,x],[z,x,y],[z,y,x]]
ov=np.unique(ov,axis=0).astype(float)

fig=make_subplots(rows=1,cols=3,specs=[[{'type':'scene'}]*3],column_widths=[1,1.5,1],
                  subplot_titles=('Кубическая ячейка','Прямоугольная ячейка','Усечённый октаэдр'),horizontal_spacing=0.05)
for v,col,mode in [(cv,1,'box'),(rv,2,'box'),(ov,3,'nn')]:
    xe,ye,ze=[],[],[]
    if mode=='box':
        for a in range(len(v)):
            for b in range(a+1,len(v)):
                if np.sum(np.abs(v[a]-v[b])>1e-9)==1:
                    xe+=[v[a,0],v[b,0],None]; ye+=[v[a,1],v[b,1],None]; ze+=[v[a,2],v[b,2],None]
    else:
        d=np.linalg.norm(v[:,None]-v[None,:],axis=2)
        e=d[d>1e-9].min()
        for a in range(len(v)):
            for b in range(a+1,len(v)):
                if abs(d[a,b]-e)<1e-6:
                    xe+=[v[a,0],v[b,0],None]; ye+=[v[a,1],v[b,1],None]; ze+=[v[a,2],v[b,2],None]
    fig.add_trace(go.Scatter3d(x=xe,y=ye,z=ze,mode='lines',line=dict(color='black',width=3),showlegend=False),1,col)
cam,ax=dict(eye=dict(x=1.7,y=1.7,z=1.2)),dict(showbackground=False,showticklabels=False,title='',showgrid=False,zeroline=False)
for s in ('scene','scene2','scene3'):
    fig.update_layout(**{s:dict(camera=cam,xaxis=ax,yaxis=ax,zaxis=ax,aspectmode='data')})
fig.update_layout(height=400,width=1100,margin=dict(l=8,r=8,t=40,b=8),showlegend=False)
fig.show()

### 2.4.2 Метод жёсткой отсечки

При расчёте межатомных взаимодействий каждая частица потенциально взаимодействует со всеми остальными частицами системы. Для больших систем прямой расчёт всех пар взаимодействий требует значительных вычислительных затрат. Чтобы ускорить вычисления, используют метод отсечки (cutoff), при котором взаимодействия между атомами учитываются только в пределах заданного радиуса $r_c$. Таким образом, если расстояние между атомами превышает $r_c$, их взаимодействие не учитывается:

$$V(r) = \begin{cases} \dfrac{q_i q_j}{4\pi\varepsilon_0\, r} & r \leq r_c \\ 0 & r > r_c \end{cases}$$

Проблема жёсткой отсечки заключается в том, что потенциальная энергия обычно не равна нулю на границе $r_c$. Поэтому при пересечении этой границы энергия скачкообразно изменяется. Сила, определяемая как отрицательная производная потенциала:

$$ \mathbf F(r)=-\nabla V(r)$$

также может испытывать разрыв. Это может нарушить устойчивость численного интегрирования. Один из способов устранения скачка энергии является использование сдвинутого потенциала:

$$
V_{\mathrm{shift}}(r)=\begin{cases}
V(r)-V(r_c), & r < r_c \\
0, & r \ge r_c.
\end{cases}
$$

Метод отсечки является важным инструментом повышения эффективности молекулярной динамики, однако выбор радиуса и способа обработки потенциала на границе отсечки должен учитывать тип взаимодействия и требуемую точность моделирования.

### 2.4.3 Метод суммирования Эвальда

Электростатические взаимодействия убывают с расстоянием медленно: кулоновский потенциал пропорционален $1/r$, а его соответствующая сила $1/r^2$. Поэтому простая отсечка на расстоянии $r_c$ может приводить к существенным ошибкам при моделировании систем с периодическими граничными условиями. Для корректного учёта дальнодействующих взаимодействий используют метод суммирования Эвальда (Ewald summation).

В периодической системе каждый заряд взаимодействует не только с другими зарядами в основной ячейке, но и со всеми их периодическими образами. Электростатическую энергию такой системы можно записать в виде суммы:

$$
V_{ij}=
\sum_{x}^{N_x}
\sum_{y}^{N_y}
\sum_{z}^{N_z}
\sum_i\sum_j{}
\frac{q_iq_j}
{|r_i- r_j|}
$$

Идея метода Эвальда - разбить медленно сходящийся ряд Кулона на два быстро сходящихся ряда и константу:

$$V = V_{real} + V_{recip} + V_{self}$$

**Реальное пространство** (короткодействующая часть, радиус $r_c$):

$$V_{real} = \frac{1}{2}\sum_{i,j}\frac{q_i q_j\,\mathrm{erfc}(\alpha r_{ij})}{r_{ij}}$$

**Обратное пространство** (длинноволновая часть, по векторам $\mathbf{k}$):

$$V_{recip} = \frac{2\pi}{V}\sum_{\mathbf k\neq 0}\frac{e^{-k^2/4\alpha^2}}{k^2}|\tilde\rho(\mathbf k)|^2$$

$$\tilde\rho(\mathbf k) = \sum_j q_j e^{i\mathbf k\cdot\mathbf r_j}$$

**Константа**:

$$ V_{self}= -\frac{\alpha}{\sqrt{\pi}} \sum_i q_i^2$$

Параметр $\alpha$ контролирует соотношение прямого и обратного взаимодействия. Сложность: $O(N^{3/2})$ при оптимальном $\alpha$.

### 2.4.4 Particle Mesh Ewald (PME)

Метод суммирования Эвальда позволяет эффективно учитывать дальнодействующие электростатические взаимодействия в периодических системах, однако прямое вычисление суммы в обратном пространстве требует значительных вычислительных затрат. **Particle Mesh Ewald (PME)** - это метод, который ускоряет данный расчёт за счёт перехода от отдельных атомных зарядов к пространственной сетке и использования быстрого преобразования Фурье (FFT).

Основная идея PME состоит в следующем:

1. **Интерполяция зарядов на сетку.** Заряды атомов распределяются по узлам трёхмерной сетки с помощью интерполяционных функций. Это позволяет представить плотность зарядов в виде сеточной функции $\rho_{\mathrm{mesh}}(\mathbf r)$.
2. **Преобразование Фурье.** С помощью FFT вычисляется преобразование сеточной плотности зарядов в обратное пространство:
   $$\rho_{\mathrm{mesh}}(\mathbf k) =\mathcal F\!\left[\rho_{\mathrm{mesh}}(\mathbf r)\right]$$

3. **Расчёт электростатического потенциала.** В обратном пространстве преобразованная плотность умножается на ядро кулоновского взаимодействия с соответствующим гауссовым множителем Эвальда.

4. **Обратное преобразование и вычисление сил.** Обратное FFT возвращает потенциал на пространственную сетку, после чего интерполяция позволяет получить электростатические силы, действующие на атомы.

Таким образом, PME сохраняет разложение Эвальда на короткодействующую часть в реальном пространстве и дальнодействующую часть в обратном пространстве, но вычисляет последнюю с помощью сеточных операций.

Для системы из $N$ атомов прямой расчёт суммы в обратном пространстве может иметь вычислительную сложность порядка $O(N^{3/2})$. При использовании PME типичная сложность составляет $O(N\log N)$, если размер сетки пропорционален числу атомов. Это делает метод подходящим для молекулярной динамики больших биомолекулярных систем.

PME широко применяется в современных пакетах молекулярной динамики, включая Gromacs, AMBER и NAMD. На практике точность расчёта определяется параметрами разложения Эвальда, размером сетки и порядком интерполяции зарядов.

# 𓅨 Глава 3. Основы работы с пакетом Gromacs

GROMACS (GROningen MAchine for Chemical Simulations) - пакет программ для моделирования молекулярной динамики, широко применяемый для изучения белков, нуклеиновых кислот, мембран, растворов и других молекулярных систем. Он позволяет подготавливать систему к расчёту, задавать параметры моделирования, выполнять молекулярную динамику и анализировать полученные траектории.

In [ ]:
# 𓅨 Инициализация GROMACS
REPO = "/home/jupyter/project/computational-chemistry-course"
GMX = "/opt/gromacs/bin/gmx"
print("GROMACS найден:")
!{GMX} --version | head -1

## 3.1 Файлы и форматы

Работа с GROMACS организована вокруг нескольких типов файлов, каждый из которых отвечает за определённую часть расчёта: описание координат и топологии системы, параметры моделирования, начальные условия и результаты вычислений.

| Расширение | Назначение | Содержимое |
|---|---|---|
| `.pdb` | Структура молекулы | Координаты атомов, названия остатков, цепи и другая структурная информация |
| `.gro` | Координаты и геометрия системы | Названия остатков и атомов, номера, координаты, параметры ячейки |
| `.top` | Основная топология | Состав системы, подключаемые параметры и количество молекул |
| `.itp` | Подключаемая топология | Описание отдельных молекул, типов атомов, связей, углов и других взаимодействий |
| `.mdp` | Параметры моделирования | Временной шаг, длительность расчёта, ансамбль, термостат, баростат и настройки взаимодействий |
| `.tpr` | Входной файл расчёта | Подготовленная бинарная система: топология, координаты, параметры и другие необходимые данные |
| `.xtc` | Траектория | Сжатые координаты атомов во времени |
| `.trr` | Траектория | Координаты, скорости и, при наличии, силы в полной точности |
| `.edr` | Энергетические данные | Кинетическая и потенциальная энергия, температура, давление и другие величины |
| `.log` | Журнал расчёта | Информация о ходе моделирования, предупреждения и диагностические сообщения |
| `.cpt` | Файл контрольной точки | Состояние расчёта, необходимое для продолжения моделирования |

## 3.2 Симуляция дипептида аланина

In [ ]:
# Готовые координаты и топология дипептида ACE-ALA-NME
main_path = os.path.join(REPO, "Занятия", "Занятие_7")
work_path = os.path.join(main_path, "dipeptide_vacuum")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

dipeptide_gro = """\
ACE-ALA-NME (аланиновый дипептид)
   22
    1ACE    CH3    1   0.244  -0.095  -0.163
    1ACE   HH31    2   0.217  -0.042  -0.244
    1ACE   HH32    3   0.343  -0.086  -0.148
    1ACE   HH33    4   0.221  -0.192  -0.177
    1ACE      C    5   0.170  -0.044  -0.043
    1ACE      O    6   0.185  -0.092   0.069
    2ALA      N    7   0.082   0.058  -0.069
    2ALA      H    8   0.075   0.093  -0.163
    2ALA     CA    9  -0.003   0.116   0.036
    2ALA     HA   10   0.053   0.117   0.119
    2ALA     CB   11  -0.039   0.259  -0.001
    2ALA    HB1   12  -0.096   0.299   0.070
    2ALA    HB2   13   0.045   0.314  -0.010
    2ALA    HB3   14  -0.089   0.260  -0.088
    2ALA      C   15  -0.129   0.030   0.059
    2ALA      O   16  -0.243   0.076   0.056
    3NME      N   17  -0.103  -0.101   0.091
    3NME      H   18  -0.009  -0.135   0.086
    3NME    CH3   19  -0.207  -0.192   0.131
    3NME   HH31   20  -0.168  -0.282   0.150
    3NME   HH32   21  -0.251  -0.157   0.214
    3NME   HH33   22  -0.275  -0.199   0.058
   0.61761   0.59536   0.45776
"""

topol_top = """\
; Include forcefield parameters
#include "amber99sb-ildn.ff/forcefield.itp"

[ moleculetype ]
; Name            nrexcl
Protein_chain_A     3

[ atoms ]
;   nr       type  resnr residue  atom   cgnr     charge       mass
; residue   1 ACE rtp ACE  q  0.0
     1         CT      1    ACE    CH3      1    -0.3662      12.01
     2         HC      1    ACE   HH31      2     0.1123      1.008
     3         HC      1    ACE   HH32      3     0.1123      1.008
     4         HC      1    ACE   HH33      4     0.1123      1.008
     5          C      1    ACE      C      5     0.5972      12.01
     6          O      1    ACE      O      6    -0.5679         16   ; qtot 0
; residue   2 ALA rtp ALA  q  0.0
     7          N      2    ALA      N      7    -0.4157      14.01
     8          H      2    ALA      H      8     0.2719      1.008
     9         CT      2    ALA     CA      9     0.0337      12.01
    10         H1      2    ALA     HA     10     0.0823      1.008
    11         CT      2    ALA     CB     11    -0.1825      12.01
    12         HC      2    ALA    HB1     12     0.0603      1.008
    13         HC      2    ALA    HB2     13     0.0603      1.008
    14         HC      2    ALA    HB3     14     0.0603      1.008
    15          C      2    ALA      C     15     0.5973      12.01
    16          O      2    ALA      O     16    -0.5679         16   ; qtot 0
; residue   3 NME rtp NME  q  0.0
    17          N      3    NME      N     17    -0.4157      14.01
    18          H      3    NME      H     18     0.2719      1.008
    19         CT      3    NME    CH3     19     -0.149      12.01
    20         H1      3    NME   HH31     20     0.0976      1.008
    21         H1      3    NME   HH32     21     0.0976      1.008
    22         H1      3    NME   HH33     22     0.0976      1.008   ; qtot 0

[ bonds ]
;  ai    aj funct
    1     2     1
    1     3     1
    1     4     1
    1     5     1
    5     6     1
    5     7     1
    7     8     1
    7     9     1
    9    10     1
    9    11     1
    9    15     1
   11    12     1
   11    13     1
   11    14     1
   15    16     1
   15    17     1
   17    18     1
   17    19     1
   19    20     1
   19    21     1
   19    22     1

[ pairs ]
;  ai    aj funct
    1     8     1
    1     9     1
    2     6     1
    2     7     1
    3     6     1
    3     7     1
    4     6     1
    4     7     1
    5    10     1
    5    11     1
    5    15     1
    6     8     1
    6     9     1
    7    12     1
    7    13     1
    7    14     1
    7    16     1
    7    17     1
    8    10     1
    8    11     1
    8    15     1
    9    18     1
    9    19     1
   10    12     1
   10    13     1
   10    14     1
   10    16     1
   10    17     1
   11    16     1
   11    17     1
   12    15     1
   13    15     1
   14    15     1
   15    20     1
   15    21     1
   15    22     1
   16    18     1
   16    19     1
   18    20     1
   18    21     1
   18    22     1

[ angles ]
;  ai    aj    ak funct
    2     1     3     1
    2     1     4     1
    2     1     5     1
    3     1     4     1
    3     1     5     1
    4     1     5     1
    1     5     6     1
    1     5     7     1
    6     5     7     1
    5     7     8     1
    5     7     9     1
    8     7     9     1
    7     9    10     1
    7     9    11     1
    7     9    15     1
   10     9    11     1
   10     9    15     1
   11     9    15     1
    9    11    12     1
    9    11    13     1
    9    11    14     1
   12    11    13     1
   12    11    14     1
   13    11    14     1
    9    15    16     1
    9    15    17     1
   16    15    17     1
   15    17    18     1
   15    17    19     1
   18    17    19     1
   17    19    20     1
   17    19    21     1
   17    19    22     1
   20    19    21     1
   20    19    22     1
   21    19    22     1

[ dihedrals ]
;  ai    aj    ak    al funct
    2     1     5     6     9
    2     1     5     7     9
    3     1     5     6     9
    3     1     5     7     9
    4     1     5     6     9
    4     1     5     7     9
    1     5     7     8     9
    1     5     7     9     9
    6     5     7     8     9
    6     5     7     9     9
    5     7     9    10     9
    5     7     9    11     9
    5     7     9    15     9
    8     7     9    10     9
    8     7     9    11     9
    8     7     9    15     9
    7     9    11    12     9
    7     9    11    13     9
    7     9    11    14     9
   10     9    11    12     9
   10     9    11    13     9
   10     9    11    14     9
   15     9    11    12     9
   15     9    11    13     9
   15     9    11    14     9
    7     9    15    16     9
    7     9    15    17     9
   10     9    15    16     9
   10     9    15    17     9
   11     9    15    16     9
   11     9    15    17     9
    9    15    17    18     9
    9    15    17    19     9
   16    15    17    18     9
   16    15    17    19     9
   15    17    19    20     9
   15    17    19    21     9
   15    17    19    22     9
   18    17    19    20     9
   18    17    19    21     9
   18    17    19    22     9

[ dihedrals ]
;  ai    aj    ak    al funct
    1     7     5     6     4
    5     9     7     8     4
    9    17    15    16     4
   15    19    17    18     4

[ system ]
; Name
ACE-ALA-NME (аланиновый дипептид)

[ molecules ]
; Compound        #mols
Protein_chain_A     1
"""

for name, text in (("dipeptide.gro", dipeptide_gro), ("topol.top", topol_top)):
    with open(name, "w") as f:
        f.write(text)

In [ ]:
# Минимизация энергии: наискорейший спуск до максимальной силы < 10 кДж/моль/нм
main_path = os.path.join(REPO, "Занятия", "Занятие_7")
work_path = os.path.join(main_path, "dipeptide_vacuum")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

em_mdp = """\
integrator  = steep        ; наискорейший спуск
emtol       = 10.0         ; критерий сходимости: максимальная сила на атоме, кДж/моль/нм
emstep      = 0.01         ; начальный шаг, нм
nsteps      = 5000

cutoff-scheme = Verlet     ; Verlet neighbor list
nstlist       = 10
coulombtype   = Cut-off
rcoulomb      = 1.4        ; молекула целиком помещается внутрь радиуса обрезания,
rvdw          = 1.4        ; а её периодические образы до неё не достают - это и есть «вакуум»
pbc           = xyz
"""
with open("em.mdp", "w") as f:
    f.write(em_mdp)

!{GMX} editconf -f dipeptide.gro -o box.gro -c -d 1.5 -bt cubic
!{GMX} grompp -f em.mdp -c box.gro -p topol.top -o em.tpr
!{GMX} mdrun -deffnm em -ntomp 4 > mdrun_em.out 2>&1
!tail -4 em.log

In [ ]:
# Молекулярная динамика: 500 пс при 300 K
main_path = os.path.join(REPO, "Занятия", "Занятие_7")
work_path = os.path.join(main_path, "dipeptide_vacuum")
os.makedirs(work_path, exist_ok=True)
os.chdir(work_path)

md_mdp = """\
integrator = md            
dt         = 0.002         ; 2 фс - допустимо, потому что связи с водородами зафиксированы
nsteps     = 250000        ; 250000 x 2 фс = 500 пс

nstxout-compressed = 500   ; координаты в .xtc каждую 1 пс
nstenergy          = 500   ; энергии в .edr каждую 1 пс
nstlog             = 5000

continuation         = no
constraints          = h-bonds
constraint-algorithm = lincs

cutoff-scheme = Verlet
nstlist       = 10
coulombtype   = Cut-off
rcoulomb      = 1.4
rvdw          = 1.4
pbc           = xyz

tcoupl   = V-rescale       ; термостат: масштабирование скоростей со стохастическим членом
tc-grps  = System
tau-t    = 0.1             ; постоянная связи с термостатом, пс
ref-t    = 300             ; целевая температура, K
pcoupl   = no              ; в вакууме давление не определено, баростата нет

gen-vel  = yes             ; стартовые скорости из распределения Максвелла
gen-temp = 300
gen-seed = -1
"""
with open("md.mdp", "w") as f:
    f.write(md_mdp)

!{GMX} grompp -f md.mdp -c em.gro -p topol.top -o md.tpr
!{GMX} mdrun -deffnm md -ntomp 4 > mdrun_md.out 2>&1
!tail -4 md.log

# 📚 Дополнительная литература
* [GROMACS Reference Manual](https://manual.gromacs.org/current/reference-manual/index.html)
* Jensen F. *Introduction to Computational Chemistry*, 3rd ed. Wiley, 2017. ISBN 978-1-118-82599-0
* Frenkel D., Smit B. *Understanding Molecular Simulation: From Algorithms to Applications*, 3rd ed. Academic Press, 2023.